# Autonomous Trailer: a Knowledge-Based Agent
**ICS 3202 Artificial Intelligence | Practical Exercise 4**

**Group members**
Abdulle Abdullahi Faduma - 191648
Eeshah Sabir khan-

Repository: `KnowledgeEngineering`

## 1. Introduction

In class we talked about a self-driving trailer that has to move cargo between places around Nairobi. In Practical Exercise 3 we modelled the area as a directed graph and used a search tree with distance measures to find routes. The problem with that version is that the agent only knew distances. It had no idea that a road is murram and turns bad in the rain, or that trailers are not allowed on some roads.

For this exercise we turn the trailer into a **knowledge-based agent**. It has three parts:

1. **Knowledge base (KB)**: facts about the world (roads, distances, surfaces, weight limits, traffic, current weather and load).
2. **Inference engine**: if-then rules, applied with forward chaining, which derive new facts such as "this road is blocked for us today".
3. **Search**: A* over the road graph, but it uses what the KB has inferred, so blocked roads are skipped and risky ones are made more expensive.

The agent can also explain why it chose a route, which is the main advantage of a knowledge-based approach over plain search.

## 2. Groupwork evidence
*(Replace this cell with the team photo or a link to the minutes PDF. If the Google Drive image trick does not work, upload the file to the repo and mention its filename here.)*

`![Team photo](teamwork_evidence.jpg)`

## 3. Building the knowledge base

In [1]:
import heapq
import math

Facts are stored as tuples, for example `("road", "Strathmore", "CBD")`. Variables in rules start with `?`.
The distances below are rough road distances in km. **Replace them with the graph and distances from Practical Exercise 3** so the two exercises stay consistent.

In [2]:
class KnowledgeBase:
    def __init__(self):
        self.facts = set()
        self.why = {}          # derived fact -> (rule name, facts it came from)

    def tell(self, fact):
        self.facts.add(fact)

    def retract(self, predicate):
        # remove every fact starting with this predicate (used when conditions change)
        self.facts = {f for f in self.facts if f[0] != predicate}

    def ask(self, *pattern):
        # return all bindings that make the pattern true
        results = []
        for f in self.facts:
            b = match(pattern, f, {})
            if b is not None:
                results.append(b)
        return results

    def holds(self, fact):
        return fact in self.facts


def is_var(x):
    return isinstance(x, str) and x.startswith("?")


def match(pattern, fact, binding):
    if len(pattern) != len(fact):
        return None
    b = dict(binding)
    for p, f in zip(pattern, fact):
        if is_var(p):
            if p in b and b[p] != f:
                return None
            b[p] = f
        elif p != f:
            return None
    return b

In [14]:
kb = KnowledgeBase()

# (a, b, km, surface, load limit in tonnes, usual traffic, trailers allowed?)
roads = [
    ("Strathmore", "CBD",        6,  "tarmac", 40, "high",   True),
    ("Strathmore", "Industrial", 5,  "tarmac", 40, "medium", True),
    ("Strathmore", "Karen",      12, "tarmac", 30, "medium", True),
    ("CBD",        "Westlands",  5,  "tarmac", 40, "high",   True),
    ("CBD",        "Industrial", 6,  "tarmac", 40, "medium", True),
    ("CBD",        "Kasarani",   11, "tarmac", 40, "high",   False),   # no trailers through town
    ("Industrial", "JKIA",       12, "tarmac", 40, "low",    True),
    ("JKIA",       "Kasarani",   14, "tarmac", 40, "medium", True),
    ("Westlands",  "Karen",      10, "tarmac", 30, "medium", True),
    ("Westlands",  "Kiambu",     14, "tarmac", 30, "medium", True),
    ("Westlands",  "Kasarani",   12, "tarmac", 40, "medium", True),
    ("Kiambu",     "Kasarani",   15, "murram", 20, "low",    True),
    ("Kasarani",   "Thika",      30, "tarmac", 40, "low",    True),
    ("Karen",      "Ngong",      8,  "murram", 15, "low",    True),
]

# approximate coordinates (lat, lon), only used for the A* heuristic
coords = {
    "Strathmore": (-1.3097, 36.8118), "CBD": (-1.2864, 36.8172),
    "Westlands": (-1.2673, 36.8110),  "Industrial": (-1.3100, 36.8500),
    "JKIA": (-1.3192, 36.9275),       "Karen": (-1.3190, 36.7100),
    "Kiambu": (-1.1714, 36.8356),     "Kasarani": (-1.2200, 36.8970),
    "Thika": (-1.0332, 37.0693),      "Ngong": (-1.3667, 36.6500),
}

for a, b, km, surface, limit, traffic, trailer_ok in roads:
    for x, y in ((a, b), (b, a)):          # roads work in both directions
        kb.tell(("road", x, y))
        kb.tell(("distance", x, y, km))
        kb.tell(("surface", x, y, surface))
        kb.tell(("load_limit", x, y, limit))
        kb.tell(("traffic", x, y, traffic))
        if not trailer_ok:
            kb.tell(("no_trailers", x, y))

# places where the trailer can refuel
for place in ("Strathmore", "Westlands", "Industrial", "Kasarani", "Thika"):
    kb.tell(("fuel_station", place))

print(len(kb.facts), "facts in the knowledge base")

147 facts in the knowledge base


## 4. Rules and the inference engine
The conditions of the trip (weather, load, time of day) are also facts. The rules combine the permanent facts with the current conditions and conclude things like `blocked` or `avoid`.

A condition that starts with `<` or `>` is a numeric comparison instead of a lookup.

In [15]:
rules = [
    ("R1: trailers are banned on this road",
     [("no_trailers", "?a", "?b")],
     ("blocked", "?a", "?b")),

    ("R2: load is heavier than the road allows",
     [("load_limit", "?a", "?b", "?lim"), ("trailer_load", "?load"), ("<", "?lim", "?load")],
     ("blocked", "?a", "?b")),

    ("R3: murram road in the rain is risky",
     [("surface", "?a", "?b", "murram"), ("weather", "rainy")],
     ("avoid", "?a", "?b")),

    ("R4: heavy traffic during peak hours slows us down",
     [("traffic", "?a", "?b", "high"), ("time_of_day", "peak")],
     ("slow", "?a", "?b")),

    ("R5: a loaded trailer should stay off murram roads",
     [("surface", "?a", "?b", "murram"), ("trailer_load", "?load"), (">", "?load", 10)],
     ("avoid", "?a", "?b")),
]


def satisfy(conditions, kb, binding):
    # yield every binding that satisfies all the conditions, one at a time
    if not conditions:
        yield binding
        return
    first, rest = conditions[0], conditions[1:]
    if first[0] in ("<", ">"):
        left = binding.get(first[1], first[1])
        right = binding.get(first[2], first[2])
        ok = left < right if first[0] == "<" else left > right
        if ok:
            yield from satisfy(rest, kb, binding)
        return
    for fact in list(kb.facts):
        b = match(first, fact, binding)
        if b is not None:
            yield from satisfy(rest, kb, b)


def fill(pattern, binding):
    return tuple(binding.get(p, p) for p in pattern)


def forward_chain(kb):
    # keep applying the rules until nothing new can be derived
    # (derived facts from an earlier run are cleared first)
    for pred in ("blocked", "avoid", "slow"):
        kb.retract(pred)
    kb.why = {}
    changed = True
    while changed:
        changed = False
        for name, conds, concl in rules:
            for b in satisfy(conds, kb, {}):
                new_fact = fill(concl, b)
                if new_fact not in kb.facts:
                    kb.facts.add(new_fact)
                    kb.why[new_fact] = name
                    changed = True

## 5. Search using the knowledge
This is A* again, like in Practical Exercise 3, with two differences. It asks the KB whether a road is blocked, and the cost of a road depends on what the rules derived (avoid = 3x, slow = 1.5x). The heuristic is straight-line distance, which is never more than the real road distance, so A* still finds the cheapest route.

In [5]:
def straight_line(a, b):
    # haversine distance in km
    lat1, lon1 = map(math.radians, coords[a])
    lat2, lon2 = map(math.radians, coords[b])
    h = math.sin((lat2 - lat1) / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * math.asin(math.sqrt(h))


def road_cost(kb, a, b):
    if kb.holds(("blocked", a, b)):
        return None
    km = kb.ask("distance", a, b, "?km")[0]["?km"]
    mult = 1
    if kb.holds(("avoid", a, b)):
        mult = 3
    elif kb.holds(("slow", a, b)):
        mult = 1.5
    return km * mult


def a_star(kb, start, goal):
    frontier = [(straight_line(start, goal), 0, start, [start])]
    best = {}
    while frontier:
        f, g, node, path = heapq.heappop(frontier)
        if node == goal:
            return path, g
        if node in best and best[node] <= g:
            continue
        best[node] = g
        for nb in kb.ask("road", node, "?next"):
            nxt = nb["?next"]
            c = road_cost(kb, node, nxt)
            if c is None:
                continue
            g2 = g + c
            heapq.heappush(frontier, (g2 + straight_line(nxt, goal), g2, nxt, path + [nxt]))
    return None, None

## 6. The agent

In [6]:
FUEL_PER_KM = 0.45      # litres per km for a loaded trailer (assumed value)


class TrailerAgent:
    def __init__(self, kb, location="Strathmore"):
        self.kb = kb
        self.location = location

    def sense(self, weather, load, time_of_day, fuel):
        # update the KB with the current conditions, then re-run the rules
        for p in ("weather", "trailer_load", "time_of_day", "fuel_litres"):
            self.kb.retract(p)
        self.kb.tell(("weather", weather))
        self.kb.tell(("trailer_load", load))
        self.kb.tell(("time_of_day", time_of_day))
        self.kb.tell(("fuel_litres", fuel))
        forward_chain(self.kb)

    def plan(self, goal):
        path, cost = a_star(self.kb, self.location, goal)
        if path is None:
            return None
        km = sum(self.kb.ask("distance", a, b, "?km")[0]["?km"] for a, b in zip(path, path[1:]))
        return {"path": path, "cost": cost, "km": km}

    def explain(self, plan):
        lines = []
        for a, b in zip(plan["path"], plan["path"][1:]):
            note = "normal road"
            for tag in ("avoid", "slow"):
                if (tag, a, b) in self.kb.why:
                    note = self.kb.why[(tag, a, b)]
            lines.append("  " + a + " -> " + b + "  (" + note + ")")
        # also say which direct roads were skipped and why
        skipped = [f for f in self.kb.why if f[0] == "blocked"]
        seen = set()
        for f in skipped:
            key = tuple(sorted((f[1], f[2])))
            if key not in seen:
                seen.add(key)
                lines.append("  [skipped] " + f[1] + " - " + f[2] + ": " + self.kb.why[f])
        return "\n".join(lines)

    def fuel_check(self, plan):
        fuel = self.kb.ask("fuel_litres", "?l")[0]["?l"]
        needed = plan["km"] * FUEL_PER_KM
        if needed <= fuel:
            return "Fuel ok: need about %.1f L, have %.1f L." % (needed, fuel)
        stops = [p for p in plan["path"] if self.kb.holds(("fuel_station", p))]
        msg = "Not enough fuel: need about %.1f L, have %.1f L." % (needed, fuel)
        if stops:
            msg += " Refuel at " + stops[0] + " on the way."
        else:
            msg += " No fuel station on this route!"
        return msg

    def go(self, goal):
        plan = self.plan(goal)
        if plan is None:
            print("No safe route from", self.location, "to", goal, "under current conditions.")
            return
        print("Route:", " -> ".join(plan["path"]))
        print("Distance: %d km (effective cost %.1f)" % (plan["km"], plan["cost"]))
        print(self.fuel_check(plan))
        print("Reasoning:")
        print(self.explain(plan))
        self.location = goal

## 7. Testing the agent
We run a few scenarios to see whether the agent changes its decision when the conditions change.

In [7]:
print("=== Scenario 1: light load, dry weather, off-peak ===")
agent = TrailerAgent(kb, "Strathmore")
agent.sense(weather="dry", load=8, time_of_day="offpeak", fuel=60)
agent.go("Kasarani")

=== Scenario 1: light load, dry weather, off-peak ===
Route: Strathmore -> CBD -> Westlands -> Kasarani
Distance: 23 km (effective cost 23.0)
Fuel ok: need about 10.3 L, have 60.0 L.
Reasoning:
  Strathmore -> CBD  (normal road)
  CBD -> Westlands  (normal road)
  Westlands -> Kasarani  (normal road)
  [skipped] Kasarani - CBD: R1: trailers are banned on this road


In [8]:
print("=== Scenario 2: heavy load (25 t), peak hours ===")
agent = TrailerAgent(kb, "Strathmore")
agent.sense(weather="dry", load=25, time_of_day="peak", fuel=60)
agent.go("Thika")

=== Scenario 2: heavy load (25 t), peak hours ===
Route: Strathmore -> CBD -> Westlands -> Kasarani -> Thika
Distance: 53 km (effective cost 58.5)
Fuel ok: need about 23.9 L, have 60.0 L.
Reasoning:
  Strathmore -> CBD  (R4: heavy traffic during peak hours slows us down)
  CBD -> Westlands  (R4: heavy traffic during peak hours slows us down)
  Westlands -> Kasarani  (normal road)
  Kasarani -> Thika  (normal road)
  [skipped] Kasarani - CBD: R1: trailers are banned on this road
  [skipped] Karen - Ngong: R2: load is heavier than the road allows
  [skipped] Kiambu - Kasarani: R2: load is heavier than the road allows


In [9]:
print("=== Scenario 3: raining, loaded trailer, going to Ngong ===")
agent = TrailerAgent(kb, "Strathmore")
agent.sense(weather="rainy", load=12, time_of_day="offpeak", fuel=60)
agent.go("Ngong")

=== Scenario 3: raining, loaded trailer, going to Ngong ===
Route: Strathmore -> Karen -> Ngong
Distance: 20 km (effective cost 36.0)
Fuel ok: need about 9.0 L, have 60.0 L.
Reasoning:
  Strathmore -> Karen  (normal road)
  Karen -> Ngong  (R3: murram road in the rain is risky)
  [skipped] Kasarani - CBD: R1: trailers are banned on this road


In [10]:
print("=== Scenario 4: low fuel ===")
agent = TrailerAgent(kb, "Strathmore")
agent.sense(weather="dry", load=8, time_of_day="offpeak", fuel=8)
agent.go("Thika")

=== Scenario 4: low fuel ===
Route: Strathmore -> CBD -> Westlands -> Kasarani -> Thika
Distance: 53 km (effective cost 53.0)
Not enough fuel: need about 23.9 L, have 8.0 L. Refuel at Strathmore on the way.
Reasoning:
  Strathmore -> CBD  (normal road)
  CBD -> Westlands  (normal road)
  Westlands -> Kasarani  (normal road)
  Kasarani -> Thika  (normal road)
  [skipped] Kasarani - CBD: R1: trailers are banned on this road


## 8. User interaction
A simple text interface. The user types the conditions and a destination, and the agent answers. Type `quit` to stop.
(We used a command-line style loop because it works in any Jupyter or Colab environment without extra setup.)

In [11]:
def chat():
    agent = TrailerAgent(kb, "Strathmore")
    places = sorted(coords)
    print("Autonomous Trailer ready. Known places:", ", ".join(places))
    while True:
        goal = input("\nWhere to? (or 'quit'): ").strip()
        if goal.lower() == "quit":
            break
        if goal not in coords:
            print("I don't know that place.")
            continue
        weather = input("Weather (dry/rainy): ").strip().lower() or "dry"
        load = float(input("Load in tonnes: ") or 0)
        tod = input("Time (peak/offpeak): ").strip().lower() or "offpeak"
        fuel = float(input("Fuel in litres: ") or 50)
        agent.sense(weather, load, tod, fuel)
        agent.go(goal)

# chat()   # uncomment to try it interactively

## 9. Extending the knowledge base with an API
The exercise encourages pulling data from an API where it is permitted. A simple extension is to read the live weather instead of typing it. Open-Meteo has a free endpoint that needs no key. We kept it as a function that falls back to manual input if there is no internet.

In [12]:
def live_weather(lat=-1.3097, lon=36.8118):
    # returns "rainy" or "dry" based on the current precipitation at Strathmore
    try:
        import requests
        url = "https://api.open-meteo.com/v1/forecast"
        r = requests.get(url, params={"latitude": lat, "longitude": lon, "current": "precipitation"}, timeout=5)
        rain = r.json()["current"]["precipitation"]
        return "rainy" if rain > 0 else "dry"
    except Exception as e:
        print("Could not get live weather (" + str(e) + "), assuming dry.")
        return "dry"

# agent.sense(weather=live_weather(), load=10, time_of_day="offpeak", fuel=50)

## 10. Discussion

**What makes this knowledge-based?** The decisions are not hard-coded into the search. Everything the agent knows is in the KB, and the rules derive new knowledge from it. If a road gets a new weight limit, we only change a fact. If the company adds a new policy, we only add a rule. The search code does not change.

**Limitations**
- The distances, limits and traffic values are estimates, not live data.
- The rules are simple: no uncertainty and no handling of contradicting facts.
- Cost multipliers (3x and 1.5x) were chosen by us and could be tuned.
- The agent does not track fuel across stops; it only warns.

**Possible improvements:** live traffic from an API, learning the multipliers from past trips, a bigger map scraped from OpenStreetMap, and handling several deliveries in one trip.

## 11. Conclusion
We extended the search-based trailer from Practical Exercise 3 into a knowledge-based agent with a fact base, forward-chaining rules and an A* planner that uses the derived knowledge. The scenarios show that the route changes with load, weather, traffic and fuel, and that the agent can say why.